# Explicit UK Biobank mentions

Measure explicit UK Biobank mentions in titles and abstracts from the full-endpoint publication parquet.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import shared_paths as P
from utils.data_analysis_00_dataset_analysis import (
    CATEGORY_PATTERNS,
    MODEL_NAMES,
    contains_pattern,
    load_publications,
    model_agreement_columns,
    normalise_bool,
    normalized_rows,
    output_dirs,
    sample_balanced,
    save_figure,
)

P.bootstrap()


In [ ]:
df = load_publications(P.SHOWCASE_PLUS)
df.shape


In [ ]:
table_dir, figure_dir = output_dirs("00_dataset_analysis_01_explicit_ukb_mentions")

summary = pd.DataFrame(
    {
        "n_publications": [len(df)],
        "n_explicit_ukb_mentions": [int(df["explicit_ukb_mention"].sum())],
        "explicit_ukb_mention_percent": [float(df["explicit_ukb_mention"].mean() * 100)],
    }
)
summary.to_csv(table_dir / "explicit_ukb_mention_summary.csv", index=False)
summary


In [ ]:
yearly = (
    df.dropna(subset=["analysis_year"])
    .groupby("analysis_year")
    .agg(
        n_publications=("id", "count"),
        n_explicit_ukb_mentions=("explicit_ukb_mention", "sum"),
    )
    .reset_index()
)
yearly["explicit_ukb_mention_percent"] = yearly["n_explicit_ukb_mentions"] / yearly["n_publications"] * 100
yearly.to_csv(table_dir / "explicit_ukb_mentions_by_year.csv", index=False)

figure, axis = plt.subplots(figsize=(11, 6))
axis.plot(yearly["analysis_year"], yearly["explicit_ukb_mention_percent"], marker="o")
axis.set(xlabel="Publication year", ylabel="Explicit UKB mention (%)", title="Explicit UK Biobank mentions by year")
axis.grid(alpha=0.25)
save_figure(figure, figure_dir / "explicit_ukb_mentions_by_year.png")
yearly.tail()
